#Regression

In [ ]:
# FFNN for Red Wine Quality Prediction, Regression

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
*******************
*******************
*******************


# 1. Load dataset
df = pd.read_csv("winequality-red.csv")

# Check column names
print(df.head())
print(df.columns)

# 2. Split features and target
X = df.drop("quality", axis=1)
y = df["quality"]

# 3. Train, validation, test split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

# 4. Standardize numerical features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 5. Build FFNN model
model = Sequential([
*******************
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(1, activation="linear")
])

# 6. Compile model
*******************

# 7. Early stopping
early_stop = EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)

# 8. Train model
history = model.fit(X_train_scaled, y_train, validation_data=(X_val_scaled, y_val),
    epochs=200, batch_size=32, callbacks=[early_stop], verbose=1)

# 9. Evaluate model
y_pred = model.predict(X_test_scaled).flatten()

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
*******************

print("Test MAE:", mae)
print("Test MSE:", mse)
print("Test RMSE:", rmse)
print("Test R2:", r2)

# 10. Example predictions
results = pd.DataFrame({"Actual Quality": y_test.values, "Predicted Quality": y_pred})
print(results.head(10))

#Classification

In [ ]:
# FFNN for Red Wine Quality Prediction, Classification

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# 1. Load dataset
df = pd.read_csv("winequality-red.csv")

# Check column names
print(df.head())
print(df.columns)

# 2. Convert quality score into classes
# 0 = Low quality
# 1 = Medium quality
# 2 = High quality
def quality_to_class(q):
    if q <= 5:
        return 0
    elif q == 6:
        return 1
    else:
        return 2

df["quality_class"] = df["quality"].apply(quality_to_class)

print(df[["quality", "quality_class"]].head())
print(df["quality_class"].value_counts())

# 3. Split features and target
X = df.drop(["quality", "quality_class"], axis=1)
y = df["quality_class"]

# 4. Train, validation, test split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# 5. Standardize numerical features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 6. Build FFNN classification model
model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(3, activation="softmax")
])

# 7. Compile model
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# 8. Early stopping
*******************

# 9. Train model
*******************


# 10. Evaluate model
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

# 11. Predictions
y_pred_prob = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_prob, axis=1)

print("Accuracy Score:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Low", "Medium", "High"]))

# 12. Example predictions
results = pd.DataFrame({
    "Actual Class": y_test.values,
    "Predicted Class": y_pred,
    "Low Probability": y_pred_prob[:, 0],
    "Medium Probability": y_pred_prob[:, 1],
    "High Probability": y_pred_prob[:, 2]
})

print(results.head(10))